In [1]:
##### Converts raw raster on ag GDP into final varaiables needed 
# pixel and subnational (vector) scale
# variables 
    # % of GDP which is ag

import os
import pandas as pd
import geopandas as gpd
import numpy as np
import rasterio
from rasterio.warp import reproject
from rasterio.enums import Resampling
from pathlib import Path
import rasterstats

In [2]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# sub-national geographies
total_geo = gpd.read_file(f"{cd}/Data/Clean/Geographies/subnational_total.shp")

# reference raster
ref_path = f"{cd}/Data/Clean/Production/total_production_USD_2020.tif"

# ag GDP raster
ag_GDP = f"{cd}/Data/Raw/Predictors/AgGDP_Ru/aggdp2010.tif"

# GDP raster
GDP = f"{cd}/Data/Raw/Predictors/GDP_Kummu/rast_gdpTot_1990_2022_5arcmin.tif" # band 21 for 2010

# save paths
pixels_pct_ag = f"{cd}/Data/Clean/Predictors/Rasters/pct_GDP_ag.tif"
vectors = f"{cd}/Data/Clean/Predictors/Vectors/pct_GDP_ag.csv"

In [3]:
#### Run resampling for pixel scale 

### PREP 
# get target grid from reference raster
with rasterio.open(ref_path) as ref:
    dst_crs       = ref.crs
    dst_transform = ref.transform
    dst_shape     = ref.shape
    dst_meta      = ref.meta.copy()


### RESAMPLE 
def resample_to_ref(src_path, band):
    dst_array = np.full(dst_shape, np.nan, dtype=np.float32)
    with rasterio.open(src_path) as src:
        reproject(
            source=rasterio.band(src, band),
            destination=dst_array,
            dst_crs=dst_crs,
            dst_transform=dst_transform,
            resampling=Resampling.sum,
            dst_nodata=np.nan,
        )
    dst_array[dst_array == -9999] = np.nan
    return dst_array

ag  = resample_to_ref(ag_GDP, 1)
total = resample_to_ref(GDP, 21)


### CALCULATE 
# compute % irrigated
with np.errstate(invalid="ignore", divide="ignore"):
    pct_ag = np.where(
        (total > 0) & ~np.isnan(total) & ~np.isnan(ag),
        ag / total,
        np.nan
    )

pct_ag = np.clip(pct_ag, 0, 1)

# save 
out_meta = dst_meta.copy()
out_meta.update(dtype="float32", count=1, nodata=-9999, compress="lzw")

out_arr = pct_ag.astype(np.float32)
out_arr[np.isnan(out_arr)] = -9999

with rasterio.open(pixels_pct_ag, "w", **out_meta) as dst:
    dst.write(out_arr, 1)

In [4]:
#### Run resampling for vector scale 

### Set-up 

# repoject GDF to match raster 
gdf_proj = total_geo.to_crs("EPSG:4326")

result = total_geo[["PROJ_ID"]].copy()

### RESAMPLE

# sum irrigation and cropland in each vector
zonal_ag  = rasterstats.zonal_stats(gdf_proj, ag_GDP, stats="sum", nodata=-9999)
zonal_total = rasterstats.zonal_stats(gdf_proj, GDP,   stats="sum", nodata=-9999)

### compute share at vector level
ag_sums  = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_ag])
total_sums = np.array([z["sum"] if z["sum"] is not None else np.nan for z in zonal_total])

with np.errstate(invalid="ignore", divide="ignore"):
    pct = np.where(
        (total_sums > 0) & ~np.isnan(total_sums) & ~np.isnan(ag_sums),
        ag_sums / total_sums,
        np.nan
    )

result["pct_GDP_ag"] = np.clip(pct, 0, 1)

### save
result.to_csv(vectors, index=False)